In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from PIL import Image
from tqdm.auto import tqdm
from typing import Dict
from pathlib import Path
import logging
import time
from prettytable import PrettyTable
from copy import deepcopy
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import densenet121
from torch.optim import Adam, lr_scheduler
import torchvision.transforms as tfms
import torchvision.transforms.functional as T

In [2]:
class CFG:
    BASE_PATH = Path("C:/Codes/Computer Vision/Xray/archive")

In [3]:
df = pd.read_csv( "archive/Data_Entry_2017.csv")

In [ ]:
df.head()

In [5]:
df = df[["Image Index", "Finding Labels"]]

In [ ]:
df['Finding Labels'].unique()

In [ ]:
df.head()

In [8]:
df['Finding Labels'] = df['Finding Labels'].str.replace("|", "&")
df = df[df['Finding Labels'].str.contains("No Finding") == False]

In [ ]:
folders = df['Finding Labels'].unique()
print(folders)

In [10]:
total = len(df)
train_df = pd.DataFrame()
valid_df = pd.DataFrame()
test_df = pd.DataFrame()

train_len = int(0.8 * total)
valid_len = int(0.1 * total)
test_len = total - train_len - valid_len

train_df = df[:train_len]
valid_df = df[train_len:train_len + valid_len]
test_df = df[train_len + valid_len:]

In [ ]:
for train_path in train_df['Image Index']:
    print(train_path)

In [ ]:
NEW_PATH = './archive/yolodataset/'

for folder in folders:
    Path(NEW_PATH + '/valid/' + folder).mkdir(parents=True, exist_ok=False)
    Path(NEW_PATH + '/test/' + folder).mkdir(parents=True, exist_ok=False)
    Path(NEW_PATH + '/train/' + folder).mkdir(parents=True, exist_ok=False)
    

In [14]:
for i in range(len(df)):
    if i < train_len:
        path = NEW_PATH + '/train/' + df.iloc[i]['Finding Labels']
    elif i < train_len + valid_len:
        path = NEW_PATH + '/valid/' + df.iloc[i]['Finding Labels']
    else:
        path = NEW_PATH + '/test/' + df.iloc[i]['Finding Labels']
    img_name = df.iloc[i]['Image Index']
    img_readpath = CFG.BASE_PATH / f"images-224/images-224/{img_name}"
    img = cv2.imread(img_readpath)
    cv2.imwrite(path + '/' + img_name, img)

In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo11l-cls.pt")  # load a pretrained model (recommended for training)

# Train the model
results = model.train(data="./Xray/archive/yolodataset", epochs=40, imgsz=224, batch=32, patience=5, device="cuda", lrf=1e-3, lr0=1e-2, workers=8)

In [ ]:
from ultralytics import YOLO

# Load a model
# model = YOLO("yolo11n-cls.pt")  # load an official model
model = YOLO("runs/classify/train9/weights/last.pt")  # load a custom model

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.top1  # top1 accuracy
metrics.top5  # top5 accuracy

In [ ]:
!yolo task=detect mode=predict model=runs/classify/train5/weights/last.pt conf=0.25 source=./archive/yolodataset/test/Atelectasis save=True